# 04 — Factor controls & deflated Sharpe (the credibility layer)

Regress the net OOS strategies on the factor panel (Newey–West errors, formation-aligned timing) and deflate the headline Sharpe for every configuration examined.

In [ ]:
# Path shim: make the repo root importable when running from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)


In [ ]:
import yaml
from src.data.synthetic import make_synthetic_panel
from src.utils.stats import rank_normalize_cross_section

cfg = yaml.safe_load(open(ROOT / "configs/config.yaml"))
panel, factors, meta = make_synthetic_panel(cfg, seed=cfg["run"]["seed"])
signal_cols = list(meta.index)
panel = rank_normalize_cross_section(panel, signal_cols)
print(panel["date"].nunique(), "months x", panel["ticker"].nunique(), "names")
meta


In [ ]:
from src.backtest.engine import run_model_backtest, comparison_table
from src.models.models import make_linear, make_lgbm
from src.models.icnet import make_icnet
wcfg, ecfg = cfg['walkforward'], cfg['evaluation']
results = {
    'elasticnet': run_model_backtest(panel, signal_cols,
        lambda: make_linear(cfg['models']['linear']), wcfg, ecfg),
    'lightgbm':  run_model_backtest(panel, signal_cols,
        lambda: make_lgbm(cfg['models']['lgbm']), wcfg, ecfg),
    'icnet':     run_model_backtest(panel, signal_cols,
        lambda: make_icnet(cfg['models']['icnet']), wcfg, ecfg),
}
comp = comparison_table(results); comp.round(3)

## Factor-controlled alpha

Low R² with surviving alpha = genuine information. Alpha that vanishes under controls = a known factor in disguise (reporting that is *also* a strong result).

In [ ]:
from src.evaluation.factor_controls import alpha_regression
import pandas as pd
pd.DataFrame([
    {'model': name, **{k: v for k, v in
        alpha_regression(r['series']['net'], factors).items()
        if k != 'betas'}}
    for name, r in results.items()
]).set_index('model').round(3)

## Deflated Sharpe (Bailey–López de Prado)

Trials = every single-signal book examined + both models. Undercounting trials is how the deflation gets gamed — count honestly.

In [ ]:
import numpy as np
from src.evaluation.deflated_sharpe import deflated_sharpe
from src.evaluation.portfolio import evaluate_signal_portfolio

single_srs = [evaluate_signal_portfolio(panel, c)['net']['monthly_sharpe']
              for c in signal_cols]
trials = [r['net']['monthly_sharpe'] for r in results.values()] + single_srs
best = comp['sharpe_net'].idxmax(); b = results[best]['net']
deflated_sharpe(b['monthly_sharpe'], b['n_months'], b['skew'],
                b['kurtosis'], trials)

**Caveat that belongs in every writeup:** synthetic Sharpes are pedagogically inflated (the betas are planted and known). On real OSAP data expect numbers an order of magnitude humbler — that is the honest result the project is designed to report.